# **Tratamento e preparação da base**

## Objetivo

Preparar a base de acidentes da PRF para a análise, considerando apenas os registros referentes ao estado da Bahia.

Este notebook parte das conclusões obtidas durante a exploração inicial dos dados, documentadas em [01_exploracao.ipynb](01_exploracao.ipynb) e em [docs/decisoes.md](../docs/decisoes.md).

## Importando as Bibliotecas que serão utilizadas:

In [1]:
import pandas as pd

## Carregando os dados de todos os anos (2019 a 2025) em um DataFrame único:

In [2]:
dfs = []  # Criando uma lista vazia para armazenar os CSVs

for ano in range(2019, 2026):  # Loop para automatizar a leitura dos arquivos de acidentes
    caminho = f'../data/raw/acidentes{ano}.csv'

    df_ano = pd.read_csv( 
        caminho,
        sep=';',  # Separador de valores
        decimal=',',  # Separador de número decimal
        encoding='latin-1'  # Tipo de codificação utilizada nos arquivos da PRF
    )

    df_ano['ano'] = ano  # Criando a coluna 'ano', com o seu valor correspondente
    dfs.append(df_ano)

df = pd.concat(dfs, ignore_index=True)  # Armazenando todas as leituras em um DF único
df["id"] = df["id"].astype("int64")  # Modificando o tipo da coluna 'id', de float64 para int64

print(df.shape)  # Quantidade de linhas x colunas no DataFrame criado
df

(1249899, 36)


,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,...,ilesos,feridos_leves,feridos_graves,mortos,latitude,longitude,regional,delegacia,uop,ano
0,182256,403856,2019-01-01,terça-feira,04:00:00,CE,116,136.9,RUSSAS,Animais na Pista,...,1,0,0,0,-4.766018,-38.056034,SPRF-CE,DEL03-CE,UOP01-DEL03-CE,2019
1,182263,402859,2019-01-01,terça-feira,05:00:00,MT,158,599.5,AGUA BOA,Defeito Mecânico no Veículo,...,1,0,0,0,-14.319000,-52.169000,SPRF-MT,DEL07-MT,UOP02-DEL07-MT,2019
2,182277,402850,2019-01-01,terça-feira,10:00:00,PA,10,34.0,DOM ELISEU,Velocidade Incompatível,...,1,0,0,0,-4.152967,-47.546082,SPRF-PA,DEL02-PA,UOP02-DEL02-PA,2019
3,182289,402431,2019-01-01,terça-feira,08:30:00,BA,101,13.0,RIO REAL,Ingestão de Álcool,...,1,0,0,0,-11.589600,-37.876100,SPRF-SE,DEL02-SE,UOP03-DEL02-SE,2019
4,182307,402638,2019-01-01,terça-feira,13:50:00,BA,116,440.0,FEIRA DE SANTANA,Ingestão de Álcool,...,0,1,0,0,-12.344000,-39.087000,SPRF-BA,DEL02-BA,UOP01-DEL02-BA,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1249894,757593,1701493,2025-12-14,domingo,13:50:00,PE,104,62.0,CARUARU,Ausência de reação do condutor,...,1,0,0,0,-8.253680,-35.975460,SPRF-PE,DEL02-PE,UOP01-DEL02-PE,2025
1249895,758175,1702860,2025-12-15,segunda-feira,15:50:00,SC,101,130.0,CAMBORIU,Condutor deixou de manter distância do veículo...,...,1,0,0,0,-26.993060,-48.660955,SPRF-SC,DEL04-SC,UOP03-DEL04-SC,2025
1249896,758175,1702859,2025-12-15,segunda-feira,15:50:00,SC,101,130.0,CAMBORIU,Condutor deixou de manter distância do veículo...,...,1,0,0,0,-26.993060,-48.660955,SPRF-SC,DEL04-SC,UOP03-DEL04-SC,2025
1249897,758186,1702906,2025-11-02,domingo,23:00:00,MG,381,218.3,BELO ORIENTE,Acessar a via sem observar a presença dos outr...,...,0,0,1,0,-19.298811,-42.371921,SPRF-MG,DEL06-MG,UOP01-DEL06-MG,2025


## Os valores da coluna `uf` estão padronizados?

Antes de filtrar os registros da Bahia, vamos verificar se os valores da coluna `uf` estão padronizados, identificar possíveis valores inválidos (ou `nulos`) e contabilizar os registros de cada estado.

**Critério de filtragem:** será utilizada a coluna `uf`, que identifica o estado do registro do acidente, e não `regional`. Durante a exploração, foi identificado ao menos um registro com `uf = BA` e `regional = SPRF-SE`, demonstrando que a regional da PRF pode pertencer a outro estado. Portanto, utilizar `regional` poderia excluir registros válidos da Bahia.

In [3]:
print(df['uf'].nunique(), 'estados diferentes')  # Quantidades de valores únicos na coluna 'uf'
df['uf'].value_counts(dropna=False)  # Conta quantas vezes cada 'uf' aparece nessa coluna

27 estados diferentes


uf
MG    168345
PR    138664
SC    137003
RJ     89616
RS     88039
SP     78940
BA     74230
GO     61346
MT     50660
PE     49752
ES     43635
MS     32345
RO     28607
PB     26875
CE     26494
RN     23757
MA     23249
PI     22609
PA     19733
DF     17807
TO     12983
AL     11192
SE      9800
AC      4508
RR      3996
AP      2905
AM      2809
Name: count, dtype: int64

**Conclusão:** A coluna `uf` apresenta 27 valores distintos, correspondentes às **27 Unidades Federativas Brasileiras**, sem valores `nulos` ou inconsistências aparentes de padronização. A Bahia (BA) possui **74.230 linhas** na base nacional, portanto podemos utilizar essa coluna para realizar o filtro.

## Criando o `df_ba`, filtrando os registros de `uf` = 'BA':

A partir da base nacional, será criado um novo DataFrame (`df_ba`) contendo apenas as linhas cuja Unidade Federativa seja **BA** (Bahia). Essa base será utilizada nas próximas etapas de tratamento e análise.

In [ ]:
df_ba = df[df['uf'] == 'BA'].copy()  # Criando um DataFrame independente apenas com UF da Bahia

df_ba.head()  # Visualizando os primeiros registros da base filtrada

,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,...,ilesos,feridos_leves,feridos_graves,mortos,latitude,longitude,regional,delegacia,uop,ano
3,182289,402431,2019-01-01,terça-feira,08:30:00,BA,101,13.0,RIO REAL,Ingestão de Álcool,...,1,0,0,0,-11.589600,-37.876100,SPRF-SE,DEL02-SE,UOP03-DEL02-SE,2019
4,182307,402638,2019-01-01,terça-feira,13:50:00,BA,116,440.0,FEIRA DE SANTANA,Ingestão de Álcool,...,0,1,0,0,-12.344000,-39.087000,SPRF-BA,DEL02-BA,UOP01-DEL02-BA,2019
5,182307,402642,2019-01-01,terça-feira,13:50:00,BA,116,440.0,FEIRA DE SANTANA,Ingestão de Álcool,...,0,0,1,0,-12.344000,-39.087000,SPRF-BA,DEL02-BA,UOP01-DEL02-BA,2019
21,182504,403174,2019-01-02,quarta-feira,10:15:00,BA,367,72.0,PORTO SEGURO,Desobediência às normas de trânsito pelo condutor,...,1,0,0,0,-16.387493,-39.481358,SPRF-BA,DEL09-BA,UOP01-DEL09-BA,2019
79,183237,405432,2019-01-03,quinta-feira,07:20:00,BA,410,6.0,RIBEIRA DO POMBAL,Deficiência ou não Acionamento do Sistema de I...,...,1,0,0,0,-10.843930,-38.576340,SPRF-BA,DEL07-BA,UOP02-DEL07-BA,2019


### Verificando a quantidade de linhas e colunas de `df_ba`, após a aplicação do filtro:

In [20]:
df_ba.shape  # Quantidade de linhas e colunas da base da Bahia

(74230, 36)

### Verificando se todas as linhas do novo DataFrame são, exclusivamente, da Bahia:

In [ ]:
df_ba['uf'].value_counts()  # Contabilizando os valores da coluna uf

uf
BA    74230
Name: count, dtype: int64

### Verificando a quantidade de linhas por ano e a soma de todas elas:

In [21]:
soma_ano = df_ba['ano'].value_counts().sort_index()  # Contabilizando as linhas por ano em ordem crescente

print(soma_ano.sum(), 'linhas totais\n\n', soma_ano)  # Conferindo o total e a distribuição anual

74230 linhas totais

 ano
2019     9897
2020     9548
2021     9508
2022     9864
2023    11054
2024    12263
2025    12096
Name: count, dtype: int64


### Verificando a consistência da UF por acidente:

Como o filtro da Bahia foi realizado pela coluna `uf`, precisamos verificar se todas as linhas de um mesmo acidente (`id`) possuem a mesma UF na base nacional.

Essa verificação evita que um acidente seja dividido entre diferentes estados durante a filtragem, caso suas linhas apresentem valores distintos na coluna `uf`.

In [26]:
print(df.groupby('id')['uf'].nunique().max(), '"UF" por "id" único')  # Agrupando as linhas por 'id' e contando os valores únicos por grupo

1 "UF" por "id" único


O resultado foi 1, indicando que cada `id` possui apenas um valor distinto de `UF` em todas as suas linhas. Portanto, o filtro `uf = 'BA'` não divide acidentes entre diferentes estados, preservando a integridade do recorte da Bahia.

### Contabilizando os acidentes únicos:

Como a base possui granularidade por pessoa, um mesmo acidente (`id`) pode aparecer em várias linhas. Portanto, a quantidade de acidentes deve ser calculada a partir dos valores distintos da coluna `id`, e não pela quantidade de linhas.

In [23]:
print(df_ba['id'].nunique(), '"ids" únicos')  # Quantidade de acidentes distintos na Bahia

25732 "ids" únicos


### Conclusão — Preparação da base nacional e recorte da Bahia

A base nacional, composta por 1.249.899 linhas e 36 colunas, foi carregada e unificada com sucesso. Após a aplicação do filtro pela coluna `uf = 'BA'`, foram obtidas 74.230 linhas correspondentes à Bahia, distribuídas entre os sete anos analisados (2019–2025).

As validações confirmaram que todos as linhas filtradas pertencem à UF da Bahia e que a distribuição anual corresponde ao total de linhas. Considerando a granularidade por pessoa, foram identificados 25.732 acidentes distintos por meio da contagem de `id`. Esse total corresponde aos acidentes que possuem pelo menos uma linha associada à Bahia (`uf = 'BA'`) na base nacional.

Além disso, foi verificado que cada acidente possui apenas uma UF em todas as suas linhas, garantindo que o filtro não divida acidentes entre estados diferentes.

A base `df_ba` será utilizada como ponto de partida para as próximas etapas de limpeza e preparação dos dados.